In [12]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import sklearn


In [13]:
from sklearn.metrics import(
    roc_auc_score,
    f1_score,
    precision_score,
    accuracy_score,
    recall_score
)

In [14]:
from torchvision.models import densenet121

In [15]:
device = torch.device("cuda")

model = densenet121(weights=None)

model.classifier = nn.Linear(
    model.classifier.in_features,
    14
)

model = model.to(device)

model.load_state_dict(
    torch.load(
        "D:/Chest_XRay_Project/models/best_model.pth"
    )
)

model.eval()

DenseNet(
  (features): Sequential(
    (conv0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (norm0): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (relu0): ReLU(inplace=True)
    (pool0): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (denseblock1): _DenseBlock(
      (denselayer1): _DenseLayer(
        (norm1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (relu1): ReLU(inplace=True)
        (conv1): Conv2d(64, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (norm2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (relu2): ReLU(inplace=True)
        (conv2): Conv2d(128, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      )
      (denselayer2): _DenseLayer(
        (norm1): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, bias=T

In [16]:
from PIL import Image
from torchvision import transforms

transform = transforms.Compose ([
 transforms.Resize((224,224)),
 transforms.Grayscale(num_output_channels=3),
 transforms.ToTensor()   
])

In [17]:
import pandas as pd
test_list = pd.read_csv("D:/NIH chest X-Ray/test_list.txt",
                        header = None,
                        names = ["Image Index"])

import ast

df3=pd.read_csv('D:/Chest_XRay_Project/notebooks/result.csv')
df3["Labels_encoded"] = df3["Labels_encoded"].apply(ast.literal_eval)


test_df = df3[
    df3["Image Index"].isin(test_list["Image Index"])
]

In [18]:
test_df.to_csv('test.csv', index=False)

In [19]:
from torch.utils.data import Dataset



class XrayDataset(Dataset):
    def __init__(self,df,transform=None):
        self.df = df.reset_index(drop = True)
        self.transform = transform 
        
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self,idx):
        row = self.df.iloc[idx]
        
        img_path = row["Image Path"]
        label = row["Labels_encoded"]
        
        image = Image.open(img_path)
        
        if self.transform:
            image = self.transform(image)
        
        label = torch.tensor(label, dtype=torch.float32)
            
        return image, label

In [20]:
from torch.utils.data import DataLoader

test_dataset = XrayDataset(test_df, transform = transform)

test_dataloader = DataLoader(test_dataset, batch_size = 32, shuffle = True, num_workers = 0, pin_memory=True )


In [21]:
all_labels = []
all_outputs = []

model.eval()

with torch.inference_mode():
    
    for images,batch_labels in test_dataloader:
        
        images = images.to(device)
        batch_labels = batch_labels.to(device)
        
        outputs = model(images)
        
        all_outputs.append(outputs.cpu())
        all_labels.append(batch_labels.cpu())
        
        

In [22]:
all_labels = torch.cat(all_labels)
all_outputs = torch.cat(all_outputs)
# Convert logits to probabilities
probabilities = torch.sigmoid(all_outputs)

# Convert to numpy
probabilities = probabilities.cpu().numpy()
labels = all_labels.numpy()


In [27]:
import json
import numpy as np

with open(
    "D:/Chest_XRay_Project/models/best_threshold.json"
) as f:
    threshold_data = json.load(f)

best_thresholds = np.array(
    list(threshold_data.values())
)

y_pred = (probabilities > best_thresholds).astype(int)

In [42]:
import pandas as pd

threshold_df = pd.DataFrame(
    threshold_data.items(),
    columns=["Disease Index", "Threshold"]
)

threshold_df.to_csv(
    "D:/Chest_XRay_Project/notebooks/outputs/metrics/best_threshold.csv",
    index=False
)

threshold_df

,Disease Index,Threshold
0,0,0.04
1,1,0.07
2,2,0.05
3,3,0.08
4,4,0.03
5,5,0.30
6,6,0.10
7,7,0.05
8,8,0.02
9,9,0.04


In [44]:
import numpy as np

best_thresholds = np.array(
    list(threshold_data.values())
)

y_pred = (probabilities > best_thresholds).astype(int)

In [46]:
print(probabilities.shape)
print(best_thresholds.shape)

(25596, 14)
(14,)


In [47]:
y_true = all_labels.numpy()
y_prob = probabilities


In [48]:
print(y_true.shape)
print(y_pred.shape)
print(probabilities.shape)

(25596, 14)
(25596, 14)
(25596, 14)


In [49]:
f1 = f1_score(
    y_true,
    y_pred,
    average = "macro"
)

precision = precision_score(
    y_true,
    y_pred,
    average = "macro"
)

recall = recall_score(
    y_true,
    y_pred,
    average = "macro"
    
)

auroc = roc_auc_score(
    y_true,
    y_prob,
    average = "macro"
)

accuracy = accuracy_score(
    y_true,
    y_pred,
)


In [ ]:
print("F1:", f1)
print("Precision:", precision)
print("Recall:", recall)
print("AUROC:", auroc)
print("Accuracy:", accuracy)




F1: 0.2805883964264764
Precision: 0.2340530909413868
Recall: 0.40301493857687376
AUROC: 0.7368687380970975
Accuracy: 0.12173777152680106


In [51]:
import json

with open("disease_mapping.json") as f:
    disease_mapping = json.load(f)

disease_names = list(disease_mapping.values())

In [52]:
print(disease_names)


['Cardiomegaly', 'Emphysema', 'Effusion', 'Hernia', 'Infiltration', 'Mass', 'Nodule', 'Atelectasis', 'Pneumothorax', 'Pleural_Thickening', 'Pneumonia', 'Fibrosis', 'Edema', 'Consolidation']


In [53]:
from sklearn.metrics import roc_auc_score


for i, disease in enumerate(disease_names):

    auc = roc_auc_score(
        all_labels[:, i],
        y_prob[:, i]
    )

    print(f"{disease}: AUROC = {auc:.4f}")

Cardiomegaly: AUROC = 0.8109
Emphysema: AUROC = 0.8529
Effusion: AUROC = 0.7800
Hernia: AUROC = 0.8238
Infiltration: AUROC = 0.6202
Mass: AUROC = 0.7631
Nodule: AUROC = 0.6946
Atelectasis: AUROC = 0.6991
Pneumothorax: AUROC = 0.7961
Pleural_Thickening: AUROC = 0.6975
Pneumonia: AUROC = 0.6220
Fibrosis: AUROC = 0.7282
Edema: AUROC = 0.7675
Consolidation: AUROC = 0.6603


In [ ]:
from sklearn.metrics import roc_auc_score
import pandas as pd
import os

auc_results = []

for i, disease in enumerate(disease_names):

    auc = roc_auc_score(
        all_labels[:, i],
        y_prob[:, i]
    )

    print(f"{disease}: AUROC = {auc:.4f}")

    auc_results.append({
        "Disease": disease,
        "AUROC": auc
    })


auc_df = pd.DataFrame(auc_results)



auc_df.to_csv(
    "../outputs/metrics/auc_scores.csv",
    index=False
)

auc_df

Cardiomegaly: AUROC = 0.8109
Emphysema: AUROC = 0.8529
Effusion: AUROC = 0.7800
Hernia: AUROC = 0.8238
Infiltration: AUROC = 0.6202
Mass: AUROC = 0.7631
Nodule: AUROC = 0.6946
Atelectasis: AUROC = 0.6991
Pneumothorax: AUROC = 0.7961
Pleural_Thickening: AUROC = 0.6975
Pneumonia: AUROC = 0.6220
Fibrosis: AUROC = 0.7282
Edema: AUROC = 0.7675
Consolidation: AUROC = 0.6603


,Disease,AUROC
0,Cardiomegaly,0.810899
1,Emphysema,0.852884
2,Effusion,0.779953
3,Hernia,0.823759
4,Infiltration,0.620199
5,Mass,0.763148
6,Nodule,0.694591
7,Atelectasis,0.699105
8,Pneumothorax,0.796104
9,Pleural_Thickening,0.697517


In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score
import matplotlib.pyplot as plt
import os


for i, disease in enumerate(disease_names):

    fpr, tpr, thresholds = roc_curve(
        all_labels[:, i],
        y_prob[:, i]
    )

    auc = roc_auc_score(
        all_labels[:, i],
        y_prob[:, i]
    )

    plt.figure(figsize=(6,5))

    plt.plot(
        fpr,
        tpr,
        label=f"AUC = {auc:.4f}"
    )

    plt.plot(
        [0,1],
        [0,1],
        linestyle="--"
    )

    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"ROC Curve - {disease}")

    plt.legend()

    plt.savefig(
        f"../outputs/plots/roc_curves/{disease}.png",
        bbox_inches="tight"
    )

    plt.close()

In [56]:
per_y_predict=(y_prob >= 0.5).astype(int)

In [57]:
per_y_predict[0]

array([0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0])

In [58]:
# 0.5 threshold

from sklearn.metrics import f1_score, precision_score, recall_score


for i, disease in enumerate(disease_names):

    f1 = f1_score(
        all_labels[:, i],
        per_y_predict[:, i]
    )

    precision = precision_score(
        all_labels[:, i],
        per_y_predict[:, i]
    )

    recall = recall_score(
        all_labels[:, i],
        per_y_predict[:, i]
    )

    print(
        f"{disease}: "
        f"F1={f1:.4f}, "
        f"Precision={precision:.4f}, "
        f"Recall={recall:.4f}"
    )

Cardiomegaly: F1=0.2024, Precision=0.4352, Recall=0.1319
Emphysema: F1=0.3301, Precision=0.4443, Recall=0.2626
Effusion: F1=0.4329, Precision=0.4850, Recall=0.3909
Hernia: F1=0.2115, Precision=0.6111, Recall=0.1279
Infiltration: F1=0.3601, Precision=0.3420, Recall=0.3802
Mass: F1=0.2999, Precision=0.2856, Recall=0.3158
Nodule: F1=0.1784, Precision=0.2543, Recall=0.1374
Atelectasis: F1=0.2690, Precision=0.3112, Recall=0.2370
Pneumothorax: F1=0.3027, Precision=0.4150, Recall=0.2383
Pleural_Thickening: F1=0.1095, Precision=0.1464, Recall=0.0875
Pneumonia: F1=0.0157, Precision=0.0610, Recall=0.0090
Fibrosis: F1=0.0493, Precision=0.1413, Recall=0.0299
Edema: F1=0.1739, Precision=0.1832, Recall=0.1654
Consolidation: F1=0.1391, Precision=0.1549, Recall=0.1262


In [59]:
# individual threshold

from sklearn.metrics import f1_score, precision_score, recall_score


for i, disease in enumerate(disease_names):

    f1 = f1_score(
        all_labels[:, i],
        y_pred[:, i]
    )

    precision = precision_score(
        all_labels[:, i],
        y_pred[:, i]
    )

    recall = recall_score(
        all_labels[:, i],
        y_pred[:, i]
    )

    print(
        f"{disease}: "
        f"F1={f1:.4f}, "
        f"Precision={precision:.4f},\n "
        f"Recall={recall:.4f}"
    )

Cardiomegaly: F1=0.3245, Precision=0.3019,
 Recall=0.3508
Emphysema: F1=0.3990, Precision=0.3668,
 Recall=0.4373
Effusion: F1=0.4828, Precision=0.3848,
 Recall=0.6479
Hernia: F1=0.3134, Precision=0.4375,
 Recall=0.2442
Infiltration: F1=0.4119, Precision=0.2819,
 Recall=0.7644
Mass: F1=0.3081, Precision=0.2624,
 Recall=0.3730
Nodule: F1=0.2268, Precision=0.1879,
 Recall=0.2859
Atelectasis: F1=0.3222, Precision=0.2345,
 Recall=0.5145
Pneumothorax: F1=0.3910, Precision=0.3086,
 Recall=0.5336
Pleural_Thickening: F1=0.1679, Precision=0.1140,
 Recall=0.3185
Pneumonia: F1=0.0690, Precision=0.0408,
 Recall=0.2234
Fibrosis: F1=0.1121, Precision=0.0783,
 Recall=0.1977
Edema: F1=0.2032, Precision=0.1511,
 Recall=0.3103
Consolidation: F1=0.1963, Precision=0.1263,
 Recall=0.4408


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import os

for i, disease in enumerate(disease_names):

    cm = confusion_matrix(
        all_labels[:, i],
        y_pred[:, i]
    )

    print(f"\n{disease}")
    print(cm)

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["Negative", "Positive"]
    )

    plt.figure(figsize=(5,5))

    disp.plot()

    plt.title(f"Confusion Matrix - {disease}")

    plt.savefig(
       f"../outputs/plots/confusion_matrix/{disease}.png",
        bbox_inches="tight"
    )

    plt.close()


Cardiomegaly
[[23660   867]
 [  694   375]]

Emphysema
[[23678   825]
 [  615   478]]

Effusion
[[16113  4825]
 [ 1640  3018]]

Hernia
[[25483    27]
 [   65    21]]

Infiltration
[[ 7581 11903]
 [ 1440  4672]]

Mass
[[22015  1833]
 [ 1096   652]]

Nodule
[[21968  2005]
 [ 1159   464]]

Atelectasis
[[16811  5506]
 [ 1592  1687]]

Pneumothorax
[[19745  3186]
 [ 1243  1422]]

Pleural_Thickening
[[21623  2830]
 [  779   364]]

Pneumonia
[[22125  2916]
 [  431   124]]

Fibrosis
[[24148  1013]
 [  349    86]]

Edema
[[23058  1613]
 [  638   287]]

Consolidation
[[18247  5534]
 [ 1015   800]]


<Figure size 500x500 with 0 Axes>

<Figure size 500x500 with 0 Axes>

<Figure size 500x500 with 0 Axes>

<Figure size 500x500 with 0 Axes>

...

<Figure size 500x500 with 0 Axes>

<Figure size 500x500 with 0 Axes>

<Figure size 500x500 with 0 Axes>

<Figure size 500x500 with 0 Axes>

<Figure size 500x500 with 0 Axes>

In [ ]:
import pandas as pd

cm_results = []

for i, disease in enumerate(disease_names):

    tn, fp, fn, tp = confusion_matrix(
        all_labels[:, i],
        y_pred[:, i]
    ).ravel()

    cm_results.append({
        "Disease": disease,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp
    })


cm_df = pd.DataFrame(cm_results)

cm_df.to_csv(
    "../outputs/metrics/confusion_matrix_values.csv",
    index=False
)

cm_df

,Disease,TN,FP,FN,TP
0,Cardiomegaly,23660,867,694,375
1,Emphysema,23678,825,615,478
2,Effusion,16113,4825,1640,3018
3,Hernia,25483,27,65,21
4,Infiltration,7581,11903,1440,4672
5,Mass,22015,1833,1096,652
6,Nodule,21968,2005,1159,464
7,Atelectasis,16811,5506,1592,1687
8,Pneumothorax,19745,3186,1243,1422
9,Pleural_Thickening,21623,2830,779,364


In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score
import pandas as pd
import os

metrics = []

for i, disease in enumerate(disease_names):

    f1 = f1_score(
        all_labels[:, i],
        y_pred[:, i]
    )

    precision = precision_score(
        all_labels[:, i],
        y_pred[:, i]
    )

    recall = recall_score(
        all_labels[:, i],
        y_pred[:, i]
    )

    print(
        f"{disease}: "
        f"F1={f1:.4f}, "
        f"Precision={precision:.4f}, "
        f"Recall={recall:.4f}"
    )

    metrics.append({
        "Disease": disease,
        "F1 Score": f1,
        "Precision": precision,
        "Recall": recall
    })


metrics_df = pd.DataFrame(metrics)


metrics_df.to_csv(
    "../outputs/metrics/f1_precision_recall.csv",
    index=False
)

metrics_df

Cardiomegaly: F1=0.3245, Precision=0.3019, Recall=0.3508
Emphysema: F1=0.3990, Precision=0.3668, Recall=0.4373
Effusion: F1=0.4828, Precision=0.3848, Recall=0.6479
Hernia: F1=0.3134, Precision=0.4375, Recall=0.2442
Infiltration: F1=0.4119, Precision=0.2819, Recall=0.7644
Mass: F1=0.3081, Precision=0.2624, Recall=0.3730
Nodule: F1=0.2268, Precision=0.1879, Recall=0.2859
Atelectasis: F1=0.3222, Precision=0.2345, Recall=0.5145
Pneumothorax: F1=0.3910, Precision=0.3086, Recall=0.5336
Pleural_Thickening: F1=0.1679, Precision=0.1140, Recall=0.3185
Pneumonia: F1=0.0690, Precision=0.0408, Recall=0.2234
Fibrosis: F1=0.1121, Precision=0.0783, Recall=0.1977
Edema: F1=0.2032, Precision=0.1511, Recall=0.3103
Consolidation: F1=0.1963, Precision=0.1263, Recall=0.4408


,Disease,F1 Score,Precision,Recall
0,Cardiomegaly,0.324535,0.301932,0.350795
1,Emphysema,0.398998,0.366846,0.437328
2,Effusion,0.482841,0.384802,0.647918
3,Hernia,0.313433,0.437500,0.244186
4,Infiltration,0.411866,0.281870,0.764398
5,Mass,0.308056,0.262374,0.372998
6,Nodule,0.226784,0.187930,0.285890
7,Atelectasis,0.322193,0.234534,0.514486
8,Pneumothorax,0.391035,0.308594,0.533583
9,Pleural_Thickening,0.167858,0.113964,0.318460
